In [1]:
# %% [markdown]
# # Notebook 02: Feature Engineering & Target Creation
#
# **Objective**:
# 1. Load the cleaned dataset from Notebook 01.
# 2. Clean DPQ columns (replace 5.397605e-79 with NaN).
# 3. Compute a depression score (sum of 9 PHQ‑9 items).
# 4. Create the target variable `menopause_status` (Premenopause / Perimenopause / Postmenopause) using RHQ variables.
# 5. Select relevant predictors (demographics, hormones, sleep, alcohol, depression).
# 6. Save the final dataset ready for modeling.
#
# **Author**: Sara YOUSSE
# **Date**: 2026-07-18

# %%
import pandas as pd
import numpy as np
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# %%
# ------------------------------
# 1. Load the cleaned dataset
# ------------------------------
data_path = "../Dataset/"
input_file = os.path.join(data_path, "cleaned_female_nhanes.pkl")

if not os.path.isfile(input_file):
    raise FileNotFoundError(f"File not found: {input_file}. Please run Notebook 01 first.")

df = pd.read_pickle(input_file)
print(f"✅ Loaded cleaned data: {df.shape[0]} rows, {df.shape[1]} columns")

# %%
# ------------------------------
# 2. Clean DPQ columns (replace 5.397605e-79 with NaN)
# ------------------------------
# This value appears to be a placeholder for missing data in this NHANES cycle.
dpq_cols = ['DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050',
            'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090']
# Replace the strange value with NaN
for col in dpq_cols:
    if col in df.columns:
        # The value might be stored as float
        df[col] = df[col].replace(5.397605e-79, np.nan)
        # Also replace 999, 7, 9 if present (NHANES missing codes)
        df[col] = df[col].replace([7, 9, 99, 999], np.nan)

print("✅ DPQ columns cleaned (5.397605e-79 → NaN)")

# %%
# ------------------------------
# 3. Compute Depression Score (PHQ‑9 total)
# ------------------------------
# Items are scored 0-3; we sum them (ignoring NaN if at least 7 items present)
# We will use min_count=7 to keep robust scores.
dpq_available = [col for col in dpq_cols if col in df.columns]
if dpq_available:
    df['depression_score'] = df[dpq_available].sum(axis=1, min_count=7)
    print(f"✅ Depression score created (based on {len(dpq_available)} items)")
else:
    print("⚠️ No DPQ columns found; depression_score will be NaN")

# %%
# ------------------------------
# 4. Create Target: Menopause Status
# ------------------------------
# Definition based on NHANES RHQ variables:
# - RHD167: Age at natural menopause (if known) → Postmenopausal
# - RHQ031: Had menstrual period in last 12 months? (1=Yes, 2=No)
# We create a 3‑class target:
#   - Premenopause: RHQ031 == 1 (still menstruating)
#   - Postmenopause: RHD167 is not NaN OR (RHQ031 == 2 AND age >= 45)
#   - Perimenopause: other cases (e.g., RHQ031 missing but no RHD167, or age <45 with no menses)
# We also use RHQ078 (reason for no period) if available, but we keep it simple.

# Initialise with NaN
df['menopause_status'] = np.nan

# Postmenopausal if RHD167 is available (age at menopause known)
df.loc[df['RHD167'].notna(), 'menopause_status'] = 3  # Post

# Premenopausal if RHQ031 == 1 (had period in last 12 months) and RHD167 missing
df.loc[(df['RHQ031'] == 1) & (df['RHD167'].isna()), 'menopause_status'] = 1  # Pre

# Postmenopausal if RHQ031 == 2 (no period) and age >= 45 (and RHD167 missing but likely post)
df.loc[(df['RHQ031'] == 2) & (df['RIDAGEYR'] >= 45) & (df['RHD167'].isna()), 'menopause_status'] = 3

# Perimenopausal: the remaining cases where we have some indication but not clear
# This includes:
# - RHQ031 == 2 but age < 45
# - RHQ031 missing but RHD167 missing and age >= 45? actually we might consider peri
# - RHQ031 == 1 but age > 55? possible late perimenopause, but we keep as pre for simplicity
# We will classify as Perimenopause (2) for those who are not clearly pre or post.
mask_unknown = df['menopause_status'].isna() & (
    (df['RHQ031'].notna() | df['RHD167'].notna() | df['RIDAGEYR'].notna())
)
# For simplicity, we assign Perimenopause to all remaining non‑NaN cases that aren't pre/post.
# But we'll restrict to those with at least age known to avoid too many.
df.loc[mask_unknown & df['RIDAGEYR'].notna(), 'menopause_status'] = 2  # Peri

# Now we have a 3-class target. Let's map to labels for convenience.
status_map = {1: 'Premenopause', 2: 'Perimenopause', 3: 'Postmenopause'}
df['menopause_label'] = df['menopause_status'].map(status_map)

print(f"✅ Menopause status created. Distribution:")
print(df['menopause_label'].value_counts())

# %%
# ------------------------------
# 5. Select Predictor Columns
# ------------------------------
# We choose variables that are likely relevant and available:
# Demographics: age, income, education
# Reproductive history: RHQ010 (age at first period), RHQ060 (hormone use), RHQ078 (reason no period)
# Hormones: LBXTST (testosterone), LBXFSH (FSH), LBXEST (estradiol), LBXSHBG (SHBG)
# Sleep: SLQ300 (hours sleep), SLQ310 (trouble sleeping)
# Alcohol: ALQ130 (average drinks per day)
# Health status: HSQ590 (self-rated health)
# Depression: depression_score

predictor_cols = [
    'RIDAGEYR',          # Age
    'INDFMPIR',          # Income-to-poverty ratio
    'DMDEDUC2',          # Education level
    'RHQ010',            # Age at first period
    'RHQ060',            # Hormone therapy use
    'RHQ078',            # Reason for no period (if applicable)
    'LBXTST',            # Total testosterone
    'LBXFSH',            # FSH (useful for menopause)
    'LBXEST',            # Estradiol
    'LBXSHBG',           # SHBG
    'SLQ300',            # Sleep hours
    'SLQ310',            # Trouble sleeping
    'ALQ130',            # Average drinks per day
    'HSQ590',            # Self-rated health
    'depression_score'   # Depression score
]

# Keep only columns that actually exist in the dataframe
existing_predictors = [col for col in predictor_cols if col in df.columns]
print(f"\n📊 Selected predictors: {existing_predictors}")

# Also keep SEQN and the target for identification
keep_cols = ['SEQN'] + existing_predictors + ['menopause_status', 'menopause_label']

# Filter the dataframe to these columns
df_final = df[keep_cols].copy()

# %%
# ------------------------------
# 6. Quick overview of final dataset
# ------------------------------
print("\n📊 Final dataset shape:", df_final.shape)
print("\nFirst 5 rows:")
print(df_final.head())

# Check missing values
missing_summary = df_final.isnull().sum()
print("\n🔍 Missing values per column:")
print(missing_summary[missing_summary > 0])

# Drop rows where target is NaN (no menopause status)
df_final = df_final.dropna(subset=['menopause_status'])
print(f"\n✅ After dropping rows with missing target: {df_final.shape[0]} rows")

# %%
# ------------------------------
# 7. Save the final dataset
# ------------------------------
output_pickle = os.path.join(data_path, "final_features_target.pkl")
df_final.to_pickle(output_pickle)
print(f"✅ Final dataset saved to: {output_pickle}")

output_csv = os.path.join(data_path, "final_features_target.csv")
df_final.to_csv(output_csv, index=False)
print(f"✅ CSV version saved to: {output_csv}")

# %%
print("\n🎉 Notebook 02 completed successfully!")
print(f"   - Final dataset shape: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
print(f"   - Target distribution:\n{df_final['menopause_label'].value_counts()}")

✅ Loaded cleaned data: 3917 rows, 124 columns
✅ DPQ columns cleaned (5.397605e-79 → NaN)
✅ Depression score created (based on 9 items)
✅ Menopause status created. Distribution:
menopause_label
Postmenopause    2484
Premenopause      819
Perimenopause     614
Name: count, dtype: int64

📊 Selected predictors: ['RIDAGEYR', 'INDFMPIR', 'DMDEDUC2', 'RHQ010', 'RHQ060', 'RHQ078', 'LBXTST', 'LBXFSH', 'LBXEST', 'LBXSHBG', 'SLQ300', 'SLQ310', 'ALQ130', 'HSQ590', 'depression_score']

📊 Final dataset shape: (3917, 18)

First 5 rows:
       SEQN  RIDAGEYR  INDFMPIR  DMDEDUC2  RHQ010  RHQ060  RHQ078  LBXTST  LBXFSH  LBXEST  LBXSHBG    SLQ300    SLQ310  ALQ130  HSQ590  depression_score  menopause_status menopause_label
0  130380.0      44.0      1.41       3.0    13.0     NaN     2.0    13.8    2.58    85.0    52.01  b'00:00'  b'08:00'     1.0     1.0      2.000000e+00               3.0   Postmenopause
1  130387.0      68.0      1.32       5.0    13.0    56.0     NaN    18.7   56.84    15.4    46.60 